# `fxbasis` — Module Demo

End-to-end walkthrough using **mock EUR/USD data** that approximates mid-2024/2025 market conditions:
- EUR/USD spot ≈ 1.0850
- USD SOFR ≈ 5.20–5.30 %
- EUR €STR ≈ 3.05–3.90 %
- Swap points calibrated to produce a negative basis of roughly −5 to −27 bps

**Sections**
1. Setup & imports
2. Build the `StaticProvider`
3. Inspect individual OIS curves (`OISCurve`)
4. Build the `FXSwapBasis` object
5. Single-tenor basis & implied rates
6. Full basis curve (`BasisCurve`) + interpolation
7. Forward basis between tenors
8. Meeting-dated OIS knots
9. Quick sensitivity: spot impact on basis


## 1 · Setup & imports

In [ ]:
%matplotlib inline
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from fxbasis import FXSwapBasis, OISCurve
from fxbasis.providers import StaticProvider
from fxbasis.utils import tenor_to_years

# Nicer pandas display
pd.options.display.float_format = "{:.4f}".format
print("Imports OK ✓")

## 2 · Mock market data & `StaticProvider`

All inputs follow Bloomberg conventions:
- OIS par rates: compounded overnight rate (decimal, e.g. 0.053 = 5.30 %)
- Swap points: raw pips with `pip_scale = 4` → divide by 10 000 to get FX adjustment
- Positive swap points ⟹ EUR trades at a forward **premium** (USD rates > EUR rates)

In [ ]:
AS_OF = datetime(2025, 5, 9, 10, 0, 0)

# ── Spot ────────────────────────────────────────────────────────────────
SPOT = {"EURUSD": 1.0850}

# ── EUR/USD swap points (raw pips, pip_scale=4) ─────────────────────────
# Positive because USD yields > EUR yields → forward EUR at premium
# Values set slightly above CIP → implied EUR rate < actual → negative basis
SWAP_POINTS = {
    "EURUSD": {
        "ON":  0.4,
        "1W":  3.1,
        "1M":  14.0,
        "3M":  47.2,
        "6M":  101.0,
        "9M":  156.6,
        "1Y":  210.6,
    }
}

PIP_SCALE = {"EURUSD": 4}

# ── OIS par rates (Bloomberg compounded convention, decimal) ─────────────
OIS_RATES = {
    "USD": {  # SOFR OIS
        "ON": 0.0530,
        "1M": 0.0528,
        "3M": 0.0520,
        "6M": 0.0505,
        "9M": 0.0490,
        "1Y": 0.0475,
    },
    "EUR": {  # €STR OIS
        "ON": 0.0390,
        "1M": 0.0385,
        "3M": 0.0365,
        "6M": 0.0340,
        "9M": 0.0320,
        "1Y": 0.0305,
    },
}

provider = StaticProvider(
    as_of=AS_OF,
    spot=SPOT,
    swap_points=SWAP_POINTS,
    pip_scale=PIP_SCALE,
    ois_rates=OIS_RATES,
    meeting_ois_rates={"USD": {}, "EUR": {}},
)
print(f"Provider created · as_of = {provider.get_as_of()}")

## 3 · Inspect the OIS curves directly

Build an `OISCurve` manually from the raw par rates to inspect discount factors, zero rates, and forward rates.

In [ ]:
# Build EUR and USD curves from raw par rates
eur_ois = OISCurve.from_par_rates(
    currency="EUR",
    as_of=AS_OF.date(),
    par_rates={tenor_to_years(t): r for t, r in OIS_RATES["EUR"].items()},
)
usd_ois = OISCurve.from_par_rates(
    currency="USD",
    as_of=AS_OF.date(),
    par_rates={tenor_to_years(t): r for t, r in OIS_RATES["USD"].items()},
)

print(eur_ois)
print(usd_ois)

In [ ]:
tenors   = ["ON", "1W", "1M", "3M", "6M", "9M", "1Y"]
times    = [tenor_to_years(t) for t in tenors]

rows = []
for tenor, t in zip(tenors, times):
    rows.append({
        "Tenor": tenor,
        "T (yrs)": round(t, 4),
        "EUR DF": eur_ois.discount_factor(t),
        "EUR rate (%)": eur_ois.rate(t) * 100,
        "USD DF": usd_ois.discount_factor(t),
        "USD rate (%)": usd_ois.rate(t) * 100,
    })

df_curves = pd.DataFrame(rows).set_index("Tenor")
df_curves.style.format({
    "T (yrs)": "{:.4f}",
    "EUR DF": "{:.6f}",
    "EUR rate (%)": "{:.3f}",
    "USD DF": "{:.6f}",
    "USD rate (%)": "{:.3f}",
})

In [ ]:
# Plot discount factor curves
t_fine = np.linspace(0.005, 1.0, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_fine, [eur_ois.discount_factor(t) for t in t_fine], label="EUR (€STR)", color="royalblue")
axes[0].plot(t_fine, [usd_ois.discount_factor(t) for t in t_fine], label="USD (SOFR)", color="tomato")
axes[0].scatter(times, [eur_ois.discount_factor(t) for t in times], color="royalblue", zorder=5)
axes[0].scatter(times, [usd_ois.discount_factor(t) for t in times], color="tomato", zorder=5)
axes[0].set_title("OIS Discount Factors")
axes[0].set_xlabel("Year Fraction")
axes[0].set_ylabel("DF")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_fine, [eur_ois.rate(t) * 100 for t in t_fine], label="EUR (€STR)", color="royalblue")
axes[1].plot(t_fine, [usd_ois.rate(t) * 100 for t in t_fine], label="USD (SOFR)", color="tomato")
axes[1].scatter(times, [eur_ois.rate(t) * 100 for t in times], color="royalblue", zorder=5)
axes[1].scatter(times, [usd_ois.rate(t) * 100 for t in times], color="tomato", zorder=5)
axes[1].set_title("OIS Par Rates")
axes[1].set_xlabel("Year Fraction")
axes[1].set_ylabel("Rate (%)")
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("EUR/USD OIS Curves — Mock Data (2025-05-09)", fontsize=13)
plt.tight_layout()
plt.show()

## 4 · Build `FXSwapBasis`

In [ ]:
eurusd = FXSwapBasis("EUR", "USD", provider)
print(eurusd)
print(f"  Spot: {eurusd.spot:.4f}")
print(f"  As-of: {eurusd.as_of}")

## 5 · Single-tenor basis & implied rate

For the **3M** tenor:
- Forward outright = Spot + SwapPoints/10000
- Implied EUR rate derived from CIP: $r^{\text{impl}}_{\text{EUR}} = \left[(1 + r_{\text{USD}} \cdot T) \cdot \frac{S}{F} - 1\right] / T$
- Basis = $(r^{\text{impl}}_{\text{EUR}} - r^{\text{actual}}_{\text{EUR}}) \times 10{,}000$ bps

In [ ]:
for tenor in ["ON", "1W", "1M", "3M", "6M", "9M", "1Y"]:
    t = tenor_to_years(tenor)
    raw_pips   = SWAP_POINTS["EURUSD"][tenor]
    fwd        = SPOT["EURUSD"] + raw_pips / 10_000
    r_impl     = eurusd.implied_rate(tenor)
    r_actual   = OIS_RATES["EUR"][tenor] if tenor in OIS_RATES["EUR"] else eur_ois.rate(t)
    basis      = eurusd.basis_bps(tenor)

    print(
        f"{tenor:>4s}  T={t:.4f}  Fwd={fwd:.4f}  "
        f"r_impl={r_impl*100:.3f}%  r_actual={r_actual*100:.3f}%  "
        f"basis={basis:+.2f} bps"
    )

## 6 · Full `BasisCurve` + interpolation

In [ ]:
bc = eurusd.curve()
print(bc)
print()
print(bc.to_series().to_string())

In [ ]:
# Interpolate onto a fine grid and query specific tenors
t_fine = np.linspace(bc.times[0], bc.times[-1], 300)
basis_fine = [bc.at(t) for t in t_fine]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_fine, basis_fine, color="steelblue", lw=2, label="PCHIP interpolation")
ax.scatter(bc.times, bc.basis_bps, color="steelblue", zorder=6, s=60, label="Knot points")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.fill_between(t_fine, basis_fine, 0, where=[b < 0 for b in basis_fine],
                alpha=0.12, color="tomato", label="Negative basis")

# Annotate knot labels
tenor_labels = ["ON", "1W", "1M", "3M", "6M", "9M", "1Y"]
for t, bps, lbl in zip(bc.times, bc.basis_bps, tenor_labels):
    ax.annotate(f"{lbl}\n{bps:.1f}", xy=(t, bps),
                xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=8.5, color="steelblue")

ax.set_title("EUR/USD Cross-Currency Basis (CIP Deviation)", fontsize=13)
ax.set_xlabel("Year Fraction")
ax.set_ylabel("Basis (bps)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7 · Forward basis between tenors

$B_{\text{fwd}}(t_1, t_2) = \dfrac{B(t_2) \cdot t_2 - B(t_1) \cdot t_1}{t_2 - t_1}$

In [ ]:
fwd_pairs = [("ON", "1M"), ("1M", "3M"), ("3M", "6M"), ("6M", "1Y")]
print(f"{'Period':<12}  {'Fwd Basis (bps)':>16}")
print("-" * 32)
for t1, t2 in fwd_pairs:
    fb = bc.forward_basis(t1, t2)
    print(f"{t1}-{t2:<6}    {fb:>+14.2f}")

## 8 · Meeting-dated OIS knots

Replace standard ON/short-end knots with central bank meeting-effective dates for a more precise short-end curve.

In [ ]:
provider_meetings = StaticProvider(
    as_of=AS_OF,
    spot=SPOT,
    swap_points=SWAP_POINTS,
    pip_scale=PIP_SCALE,
    ois_rates=OIS_RATES,
    meeting_ois_rates={
        "USD": {
            "2025-06-11": 0.0535,  # Fed FOMC effective date
            "2025-07-30": 0.0525,
        },
        "EUR": {
            "2025-06-11": 0.0388,  # ECB effective date
            "2025-07-23": 0.0375,
        },
    },
)

eurusd_meetings = FXSwapBasis("EUR", "USD", provider_meetings)
bc_meetings = eurusd_meetings.curve()

print("Basis curve WITH meeting dates:")
print(bc_meetings.to_series().to_string())
print()
print("Basis curve WITHOUT meeting dates:")
print(bc.to_series().to_string())

In [ ]:
# Side-by-side comparison
df_compare = pd.DataFrame({
    "Standard knots (bps)": bc.to_series(),
    "With CB meetings (bps)": bc_meetings.to_series(),
})
df_compare["Δ (bps)"] = df_compare["With CB meetings (bps)"] - df_compare["Standard knots (bps)"]
df_compare.style.format("{:+.3f}").bar(subset=["Δ (bps)"], align="zero", color=["tomato", "steelblue"])

## 9 · Sensitivity: how spot changes affect the basis

Everything else held constant, a stronger EUR (higher spot) reduces the USD/EUR forward ratio ⟹ the implied EUR funding rate rises ⟹ basis becomes less negative.

In [ ]:
spot_grid = np.arange(1.05, 1.16, 0.01)
tenors_of_interest = ["1M", "3M", "6M", "1Y"]
colors = ["steelblue", "tomato", "seagreen", "darkorange"]

results = {t: [] for t in tenors_of_interest}

for spot_val in spot_grid:
    p = StaticProvider(
        as_of=AS_OF,
        spot={"EURUSD": spot_val},
        swap_points=SWAP_POINTS,
        pip_scale=PIP_SCALE,
        ois_rates=OIS_RATES,
        meeting_ois_rates={"USD": {}, "EUR": {}},
    )
    fx = FXSwapBasis("EUR", "USD", p)
    for t in tenors_of_interest:
        results[t].append(fx.basis_bps(t))

fig, ax = plt.subplots(figsize=(10, 5))
for tenor, color in zip(tenors_of_interest, colors):
    ax.plot(spot_grid, results[tenor], label=tenor, color=color, lw=2)

ax.axvline(SPOT["EURUSD"], color="grey", ls="--", lw=1, label=f"Current spot ({SPOT['EURUSD']:.4f})")
ax.axhline(0, color="black", lw=0.7)
ax.set_title("EUR/USD Basis Sensitivity to Spot Rate", fontsize=13)
ax.set_xlabel("EUR/USD Spot")
ax.set_ylabel("Basis (bps)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()